In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS gold_catalog COMMENT 'Catalog for gold'")

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS gold_catalog.default")

In [0]:
from pyspark.sql.functions import col, current_timestamp

df_customer = spark.read.table("silver_catalog.default.customer_table")

# DIM CUSTOMER with Country Info
df_country = spark.read.table("silver_catalog.default.country_table")

dim_customer = df_customer.alias("c").join(
    df_country.alias("ct"),
    col("c.country") == col("ct.country_code"),
    "left"
).select(
    col("c.customer_id").alias("customer_id"),
    col("c.customer_internal_id").alias("customer_name"),
    col("c.industry_type").alias("industry_type"),
    col("ct.country_name").alias("country_name"),
    col("ct.currency_code").alias("currency_code"),
    col("c.account_created_date").alias("account_created_date"),
    col("c.is_active").alias("is_active")
).dropDuplicates(["customer_id"])

display(dim_customer)

#dim_customer.write.mode("overwrite").format("delta").saveAsTable("gold_catalog.default.dim_customer")

In [0]:
from pyspark.sql.functions import col

# i want to create the dim product table that has product_id,product_internal_id as product_name,plan_name,billing_cycle,created_date,list_price,is_active
df_product = spark.read.table("silver_catalog.default.product_table")

dim_product = df_product.select(
    col("product_id").alias("product_id"),
    col("product_internal_id").alias("product_name"),
    col("plan_name").alias("plan_name"),
    col("billing_cycle").alias("billing_cycle"),
    col("created_date").alias("created_date"),
    col("list_price").alias("list_price"),
    col("is_active").alias("is_active")
).dropDuplicates(["product_id"])

display(dim_product)

#dim_product.write.mode("overwrite").format("delta").saveAsTable("gold_catalog.default.dim_product")